In [ ]:
"""
================================================================================
LEVEL 3 - TASK 2: NEURAL NETWORK WITH TENSORFLOW/KERAS
================================================================================
Objective: Build a simple feed-forward neural network for classification
Dataset: Iris Dataset
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("="*80)
print("LEVEL 3 - TASK 2: NEURAL NETWORK CLASSIFIER")
print("="*80)

# ============================================================================
# STEP 1: SETUP PATHS
# ============================================================================

current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
main_dir = os.path.dirname(current_dir)
dataset_path = os.path.join(main_dir, 'datasets', '1) iris.csv')
output_dir = os.path.join(main_dir, 'outputs')
image_dir = os.path.join(main_dir, 'images')

os.makedirs(output_dir, exist_ok=True)
os.makedirs(image_dir, exist_ok=True)

print(f"📁 Dataset path: {dataset_path}")

# ============================================================================
# STEP 2: LOAD AND PREPARE DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 2: LOAD AND PREPARE DATA")
print("="*80)

# Load dataset
if os.path.exists(dataset_path):
    iris_df = pd.read_csv(dataset_path)
    print(f"✅ Loaded from: {dataset_path}")
else:
    alt_paths = ['../datasets/1) iris.csv', './datasets/1) iris.csv', '1) iris.csv']
    for path in alt_paths:
        if os.path.exists(path):
            iris_df = pd.read_csv(path)
            print(f"✅ Loaded from: {path}")
            break

# Encode target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(iris_df['species'])

# Features
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = iris_df[features]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📌 DATASET: {X.shape[0]} samples, {X.shape[1]} features")
print(f"📌 Classes: {label_encoder.classes_}")
print(f"📌 Training: {X_train.shape[0]} samples")
print(f"📌 Testing: {X_test.shape[0]} samples")

# ============================================================================
# STEP 3: CHECK TENSORFLOW AVAILABILITY
# ============================================================================

print("\n" + "="*80)
print("STEP 3: CHECK TENSORFLOW")
print("="*80)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.utils import to_categorical
    TF_AVAILABLE = True
    print("✅ TensorFlow is installed and ready!")
except ImportError:
    TF_AVAILABLE = False
    print("⚠️ TensorFlow not installed. Using scikit-learn MLPClassifier...")
    from sklearn.neural_network import MLPClassifier

# ============================================================================
# STEP 4: BUILD AND TRAIN NEURAL NETWORK
# ============================================================================

print("\n" + "="*80)
print("STEP 4: BUILD AND TRAIN NEURAL NETWORK")
print("="*80)

if TF_AVAILABLE:
    # One-hot encode target
    y_train_onehot = to_categorical(y_train, num_classes=3)
    y_test_onehot = to_categorical(y_test, num_classes=3)
    
    # Build model
    model = keras.Sequential([
        layers.Dense(8, activation='relu', input_shape=(4,), name='hidden_1'),
        layers.Dense(6, activation='relu', name='hidden_2'),
        layers.Dense(3, activation='softmax', name='output')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    print("\n📌 MODEL ARCHITECTURE:")
    model.summary()
    
    # Train model
    print("\n📌 Training Neural Network...")
    history = model.fit(
        X_train, y_train_onehot,
        epochs=100,
        batch_size=8,
        validation_split=0.2,
        verbose=0
    )
    print("✅ Training completed!")
    
    # Evaluate
    test_loss, test_accuracy = model.evaluate(X_test, y_test_onehot, verbose=0)
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Get training history
    train_acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    train_loss = history.history['loss']
    val_loss = history.history['val_loss']
    
else:
    # Use scikit-learn MLPClassifier
    model = MLPClassifier(hidden_layer_sizes=(8, 6), activation='relu', 
                          max_iter=200, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)
    
    # Create dummy history for plotting
    train_acc = [0.5] * 100
    val_acc = [0.5] * 100
    train_loss = [0.5] * 100
    val_loss = [0.5] * 100

print(f"\n📌 Test Accuracy: {test_accuracy:.4f}")

# ============================================================================
# STEP 5: EVALUATE
# ============================================================================

print("\n" + "="*80)
print("STEP 5: EVALUATE")
print("="*80)

print("\n📌 CONFUSION MATRIX:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n📌 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# ============================================================================
# STEP 6: VISUALIZE RESULTS
# ============================================================================

print("\n" + "="*80)
print("STEP 6: VISUALIZE RESULTS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
axes[0,0].set_title('Confusion Matrix')

# 2. Training History (if TensorFlow)
if TF_AVAILABLE:
    axes[0,1].plot(train_acc, label='Training Accuracy', linewidth=2)
    axes[0,1].plot(val_acc, label='Validation Accuracy', linewidth=2)
    axes[0,1].set_xlabel('Epoch')
    axes[0,1].set_ylabel('Accuracy')
    axes[0,1].set_title('Model Accuracy')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    axes[1,0].plot(train_loss, label='Training Loss', linewidth=2)
    axes[1,0].plot(val_loss, label='Validation Loss', linewidth=2)
    axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_ylabel('Loss')
    axes[1,0].set_title('Model Loss')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
else:
    axes[0,1].text(0.5, 0.5, 'TensorFlow not installed\nUsing scikit-learn MLP', 
                   ha='center', va='center', fontsize=14)
    axes[0,1].set_title('Training History (Not Available)')
    axes[1,0].axis('off')

# 3. Model Architecture Summary (as text)
axes[1,1].axis('off')
if TF_AVAILABLE:
    arch_text = "Neural Network Architecture:\n\n"
    arch_text += "Input Layer: 4 neurons\n"
    arch_text += "Hidden Layer 1: 8 neurons (ReLU)\n"
    arch_text += "Hidden Layer 2: 6 neurons (ReLU)\n"
    arch_text += "Output Layer: 3 neurons (Softmax)\n\n"
    arch_text += f"Total Parameters: {model.count_params()}\n\n"
    arch_text += f"Test Accuracy: {test_accuracy:.4f}"
else:
    arch_text = "MLPClassifier (scikit-learn)\n\n"
    arch_text += "Hidden Layers: (8, 6)\n"
    arch_text += "Activation: ReLU\n"
    arch_text += f"Test Accuracy: {test_accuracy:.4f}"

axes[1,1].text(0.1, 0.5, arch_text, transform=axes[1,1].transAxes,
               fontsize=12, verticalalignment='center')
axes[1,1].set_title('Model Summary')

plt.tight_layout()
plt.savefig(os.path.join(image_dir, 'neural_network_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(image_dir, 'neural_network_results.png')}")

# ============================================================================
# STEP 7: SAVE RESULTS
# ============================================================================

print("\n" + "="*80)
print("STEP 7: SAVE RESULTS")
print("="*80)

results_df = pd.DataFrame({
    'Actual': label_encoder.inverse_transform(y_test),
    'Predicted': label_encoder.inverse_transform(y_pred),
    'Correct': y_test == y_pred
})
results_df.to_csv(os.path.join(output_dir, 'neural_network_results.csv'), index=False)
print(f"✅ Saved: {os.path.join(output_dir, 'neural_network_results.csv')}")

# ============================================================================
# STEP 8: SUMMARY
# ============================================================================

print("\n" + "="*80)
print("STEP 8: SUMMARY")
print("="*80)

print(f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                    LEVEL 3 - TASK 2 COMPLETED                             ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  ✅ Neural Network Classifier                                             ║
║                                                                            ║
║  ARCHITECTURE:                                                            ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  • Input: 4 neurons (sepal_length, sepal_width, petal_length, petal_width)║
║  • Hidden 1: 8 neurons (ReLU)                                             ║
║  • Hidden 2: 6 neurons (ReLU)                                             ║
║  • Output: 3 neurons (Softmax)                                            ║
║                                                                            ║
║  PERFORMANCE:                                                             ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  • Test Accuracy: {test_accuracy:.4f}                                     ║
║                                                                            ║
║  OUTPUT FILES:                                                            ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  • outputs/neural_network_results.csv                                     ║
║  • images/neural_network_results.png                                      ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
""")

print("\n🎉 NEURAL NETWORK COMPLETED SUCCESSFULLY!")